In [4]:
import os
import json
from prophet.serialize import model_to_json
from predictor import StockPredictor

# Define your datasets
stocks = ["AMZN", "AAPL", "GOOGL", "MSFT", "UDMY", "NXE", "SPY", "CDR.WA", "EH"]
cryptos = ["BTC-USD", "ETH-USD"]

# Combine into a single list
all_tickers = stocks + cryptos

for ticker in all_tickers:
    print(f"\n{'='*60}")
    print(f" Processing {ticker} ".center(60, '='))
    print(f"{'='*60}")
    
    csv_path = f"{ticker}.csv"
    
    # Skip to the next ticker if the CSV doesn't exist
    if not os.path.exists(csv_path):
        print(f"⚠️ Warning: {csv_path} not found. Skipping {ticker}.")
        continue

    try:
        # 1. Initialize and Train
        sp = StockPredictor(ticker=ticker)
        sp.load_data(csv_path)
        sp.train() 
        
        # Optional: Generate and print the summary to verify it worked
        results = sp.predict()
        StockPredictor.print_summary(results, ticker)

        # 2. Setup Directory structure
        save_dir = f"saved_models/{ticker}"
        os.makedirs(save_dir, exist_ok=True)

        # 3. Save Prophet Model
        if sp.prophet and sp.prophet.model:
            with open(f"{save_dir}/prophet_model.json", "w") as f:
                json.dump(model_to_json(sp.prophet.model), f)

        # 4. Save LightGBM Models (Point, High, and Low for each horizon)
        if sp.lgbm:
            for horizon in sp.lgbm.models:
                sp.lgbm.models[horizon].save_model(f"{save_dir}/lgbm_{horizon}_point.txt")
                sp.lgbm.lo_models[horizon].save_model(f"{save_dir}/lgbm_{horizon}_low.txt")
                sp.lgbm.hi_models[horizon].save_model(f"{save_dir}/lgbm_{horizon}_high.txt")

        # 5. Save Pipeline Config
        config = {
            "ticker": sp.ticker,
            "chronos_size": sp.chronos_size if sp.chronos else None,
            "prophet_years": sp.prophet_years,
            "cv_scores": sp.lgbm.cv_scores if sp.lgbm else None,
            "feature_cols": sp.lgbm.feature_cols if sp.lgbm else None
        }
        with open(f"{save_dir}/pipeline_config.json", "w") as f:
            json.dump(config, f, indent=4)

        print(f"✅ All individual models saved to ./{save_dir}/")

    except Exception as e:
        print(f"❌ Error processing {ticker}: {e}")

print("\n🎉 Bulk training and saving complete!")


===================== Processing AMZN ======================
[AMZN] Loaded 7,298 rows  (1997-05-15 → 2026-05-19)

  Training pipeline for AMZN

[1/4] Engineering features…
      6,794 usable rows, 85 columns (76 feature cols)

[2/4] Training LightGBM (direct multi-horizon)…
  [LGBM] Training horizon=1w (5d)  n_samples=6794
           CV MAPE = 0.0380
  [LGBM] Training horizon=1m (21d)  n_samples=6794
           CV MAPE = 0.0786
  [LGBM] Training horizon=1y (252d)  n_samples=6794
           CV MAPE = 0.2591

[3/4] Fitting Prophet (trend + seasonality)…
  [Prophet] Fitting on 1256 rows (last 5 yrs)…


12:59:03 - cmdstanpy - INFO - Chain [1] start processing
12:59:05 - cmdstanpy - INFO - Chain [1] done processing


  [Prophet] Done. Forecast horizon: 395 cal-days.

[4/4] Loading Chronos pre-trained foundation model…
      Hardware accelerator detected: CPU
  [Chronos] Loading amazon/chronos-t5-small on cpu …


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


  [Chronos] Model loaded ✓

✓ Training complete in 36.7s

                   AMZN Forecast Summary                    
  Current price  : $259.34

  1W      ▲ $   261.78  (+0.9%)   CI: [$246.72, $283.71]
          ↳ lgbm     $   259.92  (w=0.55)
          ↳ prophet  $   269.00  (w=0.15)
          ↳ chronos  $   261.63  (w=0.30)
          LightGBM CV MAPE: 3.80%

  1M      ▲ $   275.64  (+6.3%)   CI: [$224.83, $304.65]
          ↳ lgbm     $   280.82  (w=0.55)
          ↳ prophet  $   287.47  (w=0.15)
          ↳ chronos  $   260.85  (w=0.30)
          LightGBM CV MAPE: 7.86%

  1Y      ▲ $   323.69  (+24.8%)   CI: [$34.13, $3383.88]
          ↳ lgbm     $   307.06  (w=0.35)
          ↳ prophet  $   360.41  (w=0.50)
          ↳ chronos  $   255.87  (w=0.15)
          LightGBM CV MAPE: 25.91%

✅ All individual models saved to ./saved_models/AMZN/

===================== Processing AAPL ======================
[AAPL] Loaded 11,450 rows  (1980-12-12 → 2026-05-19)

  Training pipeline for AAP

...

  [Chronos] Model loaded ✓

✓ Training complete in 31.9s

                  BTC-USD Forecast Summary                  
  Current price  : $77510.58

  1W      ▲ $ 77997.88  (+0.6%)   CI: [$71015.67, $83838.90]
          ↳ lgbm     $ 78387.58  (w=0.55)
          ↳ prophet  $ 77172.37  (w=0.15)
          ↳ chronos  $ 77700.72  (w=0.30)
          LightGBM CV MAPE: 5.92%

  1M      ▲ $ 84566.20  (+9.1%)   CI: [$66461.87, $90048.88]
          ↳ lgbm     $ 90278.75  (w=0.55)
          ↳ prophet  $ 81714.48  (w=0.15)
          ↳ chronos  $ 76313.20  (w=0.30)
          LightGBM CV MAPE: 13.68%

  1Y      ▼ $ 77004.89  (-0.7%)   CI: [$1150.81, $2766744.11]
          ↳ lgbm     $127108.91  (w=0.35)
          ↳ prophet  $ 54468.95  (w=0.50)
          ↳ chronos  $ 75837.02  (w=0.15)
          LightGBM CV MAPE: 83.71%

✅ All individual models saved to ./saved_models/BTC-USD/

==================== Processing ETH-USD ====================
[ETH-USD] Loaded 3,115 rows  (2017-11-09 → 2026-05-20)

  Train

13:07:48 - cmdstanpy - INFO - Chain [1] start processing
13:07:50 - cmdstanpy - INFO - Chain [1] done processing


  [Prophet] Done. Forecast horizon: 395 cal-days.

[4/4] Loading Chronos pre-trained foundation model…
      Hardware accelerator detected: CPU
  [Chronos] Loading amazon/chronos-t5-small on cpu …


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 


  [Chronos] Model loaded ✓

✓ Training complete in 29.6s

                  ETH-USD Forecast Summary                  
  Current price  : $2130.91

  1W      ▲ $  2310.38  (+8.4%)   CI: [$2044.95, $2782.29]
          ↳ lgbm     $  2365.65  (w=0.55)
          ↳ prophet  $  2494.29  (w=0.15)
          ↳ chronos  $  2129.28  (w=0.30)
          LightGBM CV MAPE: 6.81%

  1M      ▲ $  2826.28  (+32.6%)   CI: [$1992.24, $3159.04]
          ↳ lgbm     $  3220.91  (w=0.55)
          ↳ prophet  $  2747.41  (w=0.15)
          ↳ chronos  $  2255.77  (w=0.30)
          LightGBM CV MAPE: 15.36%

  1Y      ▲ $  2382.20  (+11.8%)   CI: [$5.65, $824623.36]
          ↳ lgbm     $  2777.96  (w=0.35)
          ↳ prophet  $  2145.14  (w=0.50)
          ↳ chronos  $  2360.35  (w=0.15)
          LightGBM CV MAPE: 71.24%

✅ All individual models saved to ./saved_models/ETH-USD/

🎉 Bulk training and saving complete!


In [5]:
results

{'1w': {'price': 2310.3848668321175,
  'low': 2044.947998046875,
  'high': 2782.289571364106,
  'pct_change': 8.422456233500801,
  'log_return': 0.08086504238826675,
  'current': 2130.909912109375,
  'sources': {'lgbm': {'price': 2365.6502907831, 'weight': 0.55},
   'prophet': {'price': 2494.287119109864, 'weight': 0.15},
   'chronos': {'price': 2129.275634765625, 'weight': 0.3}},
  'lgbm_cv_mape': 0.06814317839045622},
 '1m': {'price': 2826.281831667127,
  'low': 1992.2433471679688,
  'high': 3159.040064114129,
  'pct_change': 32.632628700356804,
  'log_return': 0.28241293012389285,
  'current': 2130.909912109375,
  'sources': {'lgbm': {'price': 3220.914376109303, 'weight': 0.55},
   'prophet': {'price': 2747.4144553642054, 'weight': 0.15},
   'chronos': {'price': 2255.767333984375, 'weight': 0.3}},
  'lgbm_cv_mape': 0.15364636088132633},
 '1y': {'price': 2382.2029918064195,
  'low': 5.6515383578673895,
  'high': 824623.3613244243,
  'pct_change': 11.792759434315595,
  'log_return': 0

## Loading the models and making predictions

In [3]:
import json
import lightgbm as lgb
import pandas as pd
from prophet.serialize import model_from_json

# Import your local classes
from predictor import StockPredictor, HORIZONS
from lgbm_model import LGBMForecaster
from prophet_model import ProphetForecaster
from chronos_model import ChronosForecaster

# 1. Load the Configuration
with open("saved_models/AAPL/pipeline_config.json", "r") as f:
    config = json.load(f)

# 2. Initialize the Predictor and Load Data
sp = StockPredictor(
    ticker=config["ticker"],
    use_chronos=bool(config["chronos_size"]),
    chronos_size=config["chronos_size"] or "small",
    prophet_years=config["prophet_years"]
)
# We still need the data to generate today's features!
sp.load_data("AAPL.csv") 

# ---------------------------------------------------------
# 3. Manually Reconstruct LightGBM
# ---------------------------------------------------------
print("Loading LightGBM models...")
sp.lgbm = LGBMForecaster(HORIZONS)
sp.lgbm.feature_cols = config["feature_cols"]
sp.lgbm.cv_scores = config["cv_scores"]

for horizon in HORIZONS.keys():
    sp.lgbm.models[horizon] = lgb.Booster(model_file=f"saved_models/AAPL/lgbm_{horizon}_point.txt")
    sp.lgbm.lo_models[horizon] = lgb.Booster(model_file=f"saved_models/AAPL/lgbm_{horizon}_low.txt")
    sp.lgbm.hi_models[horizon] = lgb.Booster(model_file=f"saved_models/AAPL/lgbm_{horizon}_high.txt")

# ---------------------------------------------------------
# 4. Manually Reconstruct Prophet
# ---------------------------------------------------------
print("Loading Prophet model...")
sp.prophet = ProphetForecaster(HORIZONS)
with open("saved_models/AAPL/prophet_model.json", "r") as f:
    sp.prophet.model = model_from_json(json.load(f))

# Crucial step: Prophet needs to re-generate its future dataframe 
# in order for the predict() method to work
max_h = max(HORIZONS.values())
calendar_days = int(max_h * 1.45) + 30 
future = sp.prophet.model.make_future_dataframe(periods=calendar_days)
sp.prophet.forecast_df = sp.prophet.model.predict(future)

# ---------------------------------------------------------
# 5. Re-initialize Chronos
# ---------------------------------------------------------
if sp.use_chronos:
    print("Loading Chronos from HuggingFace...")
    import torch
    if torch.cuda.is_available():
        target_device = "cuda"
    elif torch.backends.mps.is_available():
        target_device = "mps"
    else:
        target_device = "cpu"
        
    sp.chronos = ChronosForecaster(model_size=config["chronos_size"])
    sp.chronos.load(device=target_device, verbose=False)

# ---------------------------------------------------------
# 6. Run the Prediction
# ---------------------------------------------------------
# Trick the predictor into thinking it ran the train() method
sp._trained = True 

print("\nGenerating ensemble prediction...")
results = sp.predict()
StockPredictor.print_summary(results, sp.ticker)

[AAPL] Loaded 11,450 rows  (1980-12-12 → 2026-05-19)
Loading LightGBM models...
Loading Prophet model...
Loading Chronos from HuggingFace...


We recommend keeping prediction length <= 64. The quality of longer predictions may degrade since the model is not optimized for it. 



Generating ensemble prediction...

                   AAPL Forecast Summary                    
  Current price  : $298.97

  1W      ▼ $   293.87  (-1.7%)   CI: [$274.17, $309.13]
          ↳ lgbm     $   293.41  (w=0.55)
          ↳ prophet  $   286.46  (w=0.15)
          ↳ chronos  $   298.49  (w=0.30)
          LightGBM CV MAPE: 4.19%

  1M      ▼ $   286.55  (-4.2%)   CI: [$273.77, $317.53]
          ↳ lgbm     $   281.72  (w=0.55)
          ↳ prophet  $   287.62  (w=0.15)
          ↳ chronos  $   295.06  (w=0.30)
          LightGBM CV MAPE: 9.32%

  1Y      ▲ $   347.23  (+16.1%)   CI: [$48.87, $2548.35]
          ↳ lgbm     $   310.87  (w=0.35)
          ↳ prophet  $   400.24  (w=0.50)
          ↳ chronos  $   279.91  (w=0.15)
          LightGBM CV MAPE: 34.46%

